# RT-DETR Res50 Plot Notebook (Shareable)

This notebook reads the RT-DETR training `log.txt` (JSON lines) and generates a 6-panel training figure:
- Box Loss
- Object Loss (GIoU)
- Class Loss (VFL)
- Precision (AP@0.50)
- Recall (AR@100)
- mAP@0.50:0.95

In [ ]:
# Optional: install plotting dependency if needed
# !pip install matplotlib

from pathlib import Path
import json
import matplotlib.pyplot as plt

In [ ]:
# Update these paths if your workspace location is different
LOG_FILE = Path('/home/parvezdev/fydp/outputs/backbone_benchmark/res50_coco8_train/log.txt')
OUT_IMG = Path('/home/parvezdev/fydp/outputs/backbone_benchmark/res50_coco8_train/rtdetr_res50_training_curves_from_notebook.png')
TITLE = 'RT-DETR Res50 Backbone on coco8'

if not LOG_FILE.exists():
    raise FileNotFoundError(f'Log file not found: {LOG_FILE}')

rows = []
for line in LOG_FILE.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line:
        continue
    try:
        obj = json.loads(line)
    except json.JSONDecodeError:
        continue
    if isinstance(obj, dict) and 'epoch' in obj:
        rows.append(obj)

rows.sort(key=lambda x: int(x.get('epoch', -1)))
if not rows:
    raise RuntimeError(f'No epoch rows found in: {LOG_FILE}')

def extract_series(data, key):
    xs, ys = [], []
    for r in data:
        v = r.get(key)
        if isinstance(v, (int, float)):
            xs.append(int(r['epoch']))
            ys.append(float(v))
    return xs, ys

def extract_coco_index(data, idx):
    xs, ys = [], []
    for r in data:
        vals = r.get('test_coco_eval_bbox')
        if isinstance(vals, list) and len(vals) > idx and isinstance(vals[idx], (int, float)):
            xs.append(int(r['epoch']))
            ys.append(float(vals[idx]))
    return xs, ys

def plot_panel(ax, x, y, title, ylabel, color):
    if not x:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)
        return
    ax.plot(x, y, color=color, linewidth=1.8, marker='o', markersize=3)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)

box_x, box_y = extract_series(rows, 'train_loss_bbox')
obj_x, obj_y = extract_series(rows, 'train_loss_giou')
cls_x, cls_y = extract_series(rows, 'train_loss_vfl')

# COCO vector indices:
# 0 -> AP@[0.50:0.95], 1 -> AP@0.50, 8 -> AR@100
prec_x, prec_y = extract_coco_index(rows, 1)
rec_x, rec_y = extract_coco_index(rows, 8)
map_x, map_y = extract_coco_index(rows, 0)

fig, axes = plt.subplots(3, 2, figsize=(12, 14))
fig.suptitle(TITLE, fontsize=16, fontweight='bold')

plot_panel(axes[0, 0], box_x, box_y, 'Box Loss', 'Loss', '#1f77b4')
plot_panel(axes[0, 1], prec_x, prec_y, 'Precision (AP@0.50)', 'Score', '#2ca02c')
plot_panel(axes[1, 0], obj_x, obj_y, 'Object Loss (GIoU)', 'Loss', '#ff7f0e')
plot_panel(axes[1, 1], rec_x, rec_y, 'Recall (AR@100)', 'Score', '#d62728')
plot_panel(axes[2, 0], cls_x, cls_y, 'Class Loss (VFL)', 'Loss', '#9467bd')
plot_panel(axes[2, 1], map_x, map_y, 'mAP@0.50:0.95', 'Score', '#8c564b')

plt.tight_layout(rect=[0, 0, 1, 0.97])
OUT_IMG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_IMG, dpi=180)
print(f'Saved: {OUT_IMG}')
print(f'Epoch rows: {len(rows)}')
plt.show()

In [ ]:
# Optional: quick command-style cell to run your shell script from notebook
# !bash /home/parvezdev/fydp/run_rtdetr_res50_train_and_plot.sh